# Huggingface Tutorial:

Um ein vortrainiertes Modell verwenden zu können, müssen wir zunächst ein paar libraries installieren.
Bemerkung: Huggingface bietet einige Tutorials an, die nützlich sind, um mit Transformers etc. vertraut zu werden. Was du allerdings beachten solltest, ist, dass einige Skripte so nicht in Colab laufen.
Anstelle von **!pip install transformers datasets** solltest du also folgende Befehle verwenden.
Außerdem werden oft accelerate und evaluate benötigt. Colab gibt dann eine Fehlermeldung aus, die besagt, dass man **!pip install accelerate -U** verwenden soll, allerdings lautet der richtige Befehl **!pip install -U accelerate**, andernfalls bleibt das Problem bestehen.

In [ ]:
!pip install -U transformers
!pip install -U datasets
!pip install -U accelerate
!pip install -U evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 12.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.41.2
    Uninstalling transformers-4.41.2:
      Successfully uninstalled transformers-4.41.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Unins

# Daten vorbereiten (Preprocessing)
Bevor wir unser Modell auf einen Datensatz trainieren können, müssen wir die Daten so vorbereiten, dass sie dem Eingabeformat des Modells entsprechen.
Konkret müssen wir unsere Daten also in Tensoren umwandeln (--> batches von Tensoren)

Huggingface bietet einen eigenen Tokenizer, um Text in Tokens umzuwandeln, die Tokens numerisch darzustellen und sie zu Tensoren zusammenzusetzten.

Als erstes installieren wir datasets, um auf die datasets von Huggingface zugreifen zu können (siehe oben).



Ein Tokenizer dient dazu unseren text in Tokens einzuteilen. Die Tokens werden dann zu Zahlen und schließlich zu Tensoren konvertiert, die die Eingabe unseres Modells darstellen.

Bemerkung: Wir verwenden ein vortrainiertes Modell, d.h. wir müssen darauf achten den richtigen Tokenizer auszuwählen, da verschiedene Modelle verschiedene Tokenizer verwenden. Wenn wir einen falschen Tokenizer für unser Modell verwenden, könnte es sein, dass der Text unterschiedlich tokenisiert wird und andere Indices verwendet werden.

Am einfachsten ist es den AutoTokenizer zu verwenden. Dieser sucht automatisch den richtigen Tokenizer für unser Modell heraus.
Die Methode from_pretrained() sorgt dafür, dass das Vokabular unseres vortrainierten Modells geladen wird.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
encoded_input = tokenizer("Do not meddle in the affairs of wizards, for they are subtle and quick to anger.")
print(encoded_input)

{'input_ids': [101, 2091, 1136, 1143, 13002, 1107, 1103, 5707, 1104, 16678, 1116, 117, 1111, 1152, 1132, 11515, 1105, 3613, 1106, 4470, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


Der Tokenizer dient als Encoder. Das heißt, wir bekommen einen Dictionary mit drei verschiedenen Items:
1. input_ids: jedem Token wird eine ID gegeben
2. attention_mask: gibt an, ob ein Token beachtet werden soll oder nicht
3. token_type_ids: gibt an, zu welcher Sequenz ein Token gehört, wenn es mehr als eine Sequenz gibt

Mit dem Befehl tokenizer.decode() können wir uns wieder unseren Satz ausgeben lassen:

In [ ]:
tokenizer.decode(encoded_input["input_ids"])

'[CLS] Do not meddle in the affairs of wizards, for they are subtle and quick to anger. [SEP]'


Jenachdem welchen Tokenizer man verwendet, kann es vorkommen, dass spezielle Tokens in die Sequenz hinzugefügt werden. Hier werden z.B. CLS und SEP hinzugefügt, was für classifier und seperator steht.


Wenn du mehrere Sätze vorverarbeiten möchtest, kannst du diese als Liste dem Tokenizer übergeben:

In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_inputs = tokenizer(batch_sentences)
print(encoded_inputs)

{'input_ids': [[101, 1252, 1184, 1164, 1248, 6462, 136, 102], [101, 1790, 112, 189, 1341, 1119, 3520, 1164, 1248, 6462, 117, 21902, 1643, 119, 102], [101, 1327, 1164, 5450, 23434, 136, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1]]}


# Padding

Wie du sehen kannst, sind unsere Sätze nicht alle gleich lang. Das kann zu einem Problem führen, weil unsere Tensoren (unsere Eingabe im Modell) eine einheitliche Shape brauchen.
Um dafür zu sorgen, dass alle Sätze die gleiche Länge haben, verwenden wir Padding. Wir fügen also den kürzeren Sätzen spezielle Tokens hinzu, damit sie die gleiche Länge wie der längste Satz in unserem Batch haben.


In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True)
print(encoded_input)

{'input_ids': [[101, 1252, 1184, 1164, 1248, 6462, 136, 102, 0, 0, 0, 0, 0, 0, 0], [101, 1790, 112, 189, 1341, 1119, 3520, 1164, 1248, 6462, 117, 21902, 1643, 119, 102], [101, 1327, 1164, 5450, 23434, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]}



# Truncation
Manchmal kann es auch vorkommen, dass unsere Sätze zu lang sind, als dass unser Modell sie verarbeiten kann. In diesem Fall kürzen wir die Sätze.
Wenn wir truncation=True auswählen, werden die Sätze automatisch auf die maximale Länge, die unser Modell verarbeiten kann, gekürzt.


In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True, truncation=True)
print(encoded_input)

{'input_ids': [[101, 1252, 1184, 1164, 1248, 6462, 136, 102, 0, 0, 0, 0, 0, 0, 0], [101, 1790, 112, 189, 1341, 1119, 3520, 1164, 1248, 6462, 117, 21902, 1643, 119, 102], [101, 1327, 1164, 5450, 23434, 136, 102, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]]}


# Tensoren erstellen

Schließlich müssen wir unsere Tokens noch zu Tensoren umwandeln, die wir als Engabe in unser Modell geben können.
Um Pytorch-Tensoren zu erhalten, schreiben wir **return_tensors="pt"**


In [ ]:
batch_sentences = [
    "But what about second breakfast?",
    "Don't think he knows about second breakfast, Pip.",
    "What about elevensies?",
]
encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[  101,  1252,  1184,  1164,  1248,  6462,   136,   102,     0,     0,
             0,     0,     0,     0,     0],
        [  101,  1790,   112,   189,  1341,  1119,  3520,  1164,  1248,  6462,
           117, 21902,  1643,   119,   102],
        [  101,  1327,  1164,  5450, 23434,   136,   102,     0,     0,     0,
             0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


# Vortrainiertes Modell fine-tunen
Wir verwenden ein vortrainiertes Modell, um es auf unseren (für unsere Aufgabe spezifischen) Datensatz zu trainieren = fine-tuning

Huggingface bietet dazu einen eigenen Trainer.


Bevor wir unser Modell fine-tunen können, brauchen wir einen Datensatz, den wir vorverarbeiten müssen (wie oben gesehen).

Wir wählen hier den Yelp Reviews Datensatz:

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")
print(dataset)
dataset["train"][100]
dataset["train"][10]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

{'label': 0,
 'text': "Owning a driving range inside the city limits is like a license to print money.  I don't think I ask much out of a driving range.  Decent mats, clean balls and accessible hours.  Hell you need even less people now with the advent of the machine that doles out the balls.  This place has none of them.  It is april and there are no grass tees yet.  BTW they opened for the season this week although it has been golfing weather for a month.  The mats look like the carpet at my 107 year old aunt Irene's house.  Worn and thread bare.  Let's talk about the hours.  This place is equipped with lights yet they only sell buckets of balls until 730.  It is still light out.  Finally lets you have the pit to hit into.  When I arrived I wasn't sure if this was a driving range or an excavation site for a mastodon or a strip mining operation.  There is no grass on the range. Just mud.  Makes it a good tool to figure out how far you actually are hitting the ball.  Oh, they are cash 

In [ ]:
dataset["train"].features

{'label': ClassLabel(names=['1 star', '2 star', '3 stars', '4 stars', '5 stars'], id=None),
 'text': Value(dtype='string', id=None)}

In [ ]:
dataset["train"].features[f"label"]

ClassLabel(names=['1 star', '2 star', '3 stars', '4 stars', '5 stars'], id=None)

In [ ]:
dataset["train"]["text"][:2]

["dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first.  really, what more do you need?  i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank.",
 "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patient

In [ ]:
dataset["train"]["label"][:2]

[4, 1]

Wir benötigen jetzt einen Tokenizer und verwenden Padding und Truncation (siehe oben).
Um unseren Datensatz auf einmal zu verarbeiten, können wir die Map-Funktion verweden. Sie sorgt dafür, dass unser Preprocessing auf den gesamten Datensatz angewendet wird.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]


Wir kreieren hier ein Subset, um zeiteffizienter arbeiten zu können:

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

# Training
1. mit PyTorch Trainer

Huggingface bietet eine Trainer Klasse. Diese erspart uns einen eigenen Trainingsloop zu schreiben.

Zunächst laden wir unser Modell und spezifizieren die Anzahl der Labels. Wie viele Labels ein Datenset hat, kann in der dataset card eingesehen werden.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.




Du wirst wahrscheinlich eine Warnung sehen, dass einige der trainierten Gewichte nicht verwendet werden und einige Gewichte zufällig initialisiert werden. Keine Sorge, das ist normal, denn der vorher trainierte Head des BERT-Modells wird verworfen und durch einen zufällig initialisierten Klassifikationskopf ersetzt.

Als Nächstes erstellen wir eine Klasse TrainingArguments, die alle Hyperparameter enthält, die wir einstellen können (wir verwenden hier die Standard-Trainingshyperparametern).


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir="test_trainer")

# Evaluierung
Als nächstes wollen wir unser Modell evaluieren:

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

Wir wollen nun die Genauigkeit unserer Vorhersagen berechnen. Dazu müssen wir unsere Vorhersagen in Logits umwandeln, bevor wir sie an compute übergeben.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Als evaluation_strategy wählen wir epoch, um unsere Bewertungsmetriken während des fine-tuning zu überwachen (am Ende jeder Epoche).

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Trainer
 Wir können nun ein Trainer-Objekt mit unserem Modell, den Trainingsargumenten, den Trainings- und Testdatensätzen und der Bewertungsfunktion erstellen:

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)


Dann fine-tunen/trainieren wir unser Modell:

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.094752,0.526000
2,No log,1.101937,0.552000
3,No log,1.042880,0.601000


TrainOutput(global_step=375, training_loss=1.0004009602864583, metrics={'train_runtime': 373.9111, 'train_samples_per_second': 8.023, 'train_steps_per_second': 1.003, 'total_flos': 789354427392000.0, 'train_loss': 1.0004009602864583, 'epoch': 3.0})

# 2. Training in native PyTorch

Alternativ können wir unser Modell auch mit Pytorch trainieren.



Dafür müssen wir nun einen eigenen Trainingsloop schreiben.
Bemerkung: wir geben hier Speicher frei (alternativ Neustart)


In [ ]:
import torch
del model
del trainer
torch.cuda.empty_cache()

NameError: name 'model' is not defined

Als nächstes müssen wir unser tokenized_dataset etwas nachbereiten, damit wir es trainieren können.

1. Wir müssen die Text-Spalte entfernen, da unser Modell keinen Text als Eingabe akzeptiert.
2. Wir benennen die Spalte label in labels um.
3. Wir wollen Pytorch-Tensoren ausgeben und keine Listen, weshalb wir **tokenized_datasets.set_format("torch") ** verwenden.


In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

# DataLoader
Wir benötigen nun einen DataLoader für unsere Trainings- und Testdatensätze, damit wir über Batches iterieren können:

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(small_train_dataset, shuffle=True, batch_size=8)
eval_dataloader = DataLoader(small_eval_dataset, batch_size=8)

print(train_dataloader)
next(iter(train_dataloader))
print(eval_dataloader)
next(iter(eval_dataloader))

{'labels': tensor([2, 4, 1, 4, 3, 4, 2, 3]),
 'input_ids': tensor([[  101, 14812, 16442,  ...,     0,     0,     0],
         [  101, 19383,  1303,  ...,     0,     0,     0],
         [  101, 12008, 27788,  ...,     0,     0,     0],
         ...,
         [  101,  3930, 13991,  ...,     0,     0,     0],
         [  101,  1284,  3523,  ...,     0,     0,     0],
         [  101,  6682,  3537,  ...,     0,     0,     0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])}

In [ ]:
for batch in eval_dataloader:
    print("Batch:")
    for key, value in batch.items():
        print(f"{key}: {value.shape}")
    print("------------")
    # Print only the first batch
    break

Batch:
labels: torch.Size([8])
input_ids: torch.Size([8, 512])
token_type_ids: torch.Size([8, 512])
attention_mask: torch.Size([8, 512])
------------



Wir laden unser Modell und bestimmen die Anzahl der Labels (hier: 5)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**Optimierer**
Wir benötigen jetzt noch einen Optimierer und einen Learning Rate Scheduler, um unser Modell zu fine-tunen.
Wir verwenden hier den AdamW optimizer.


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

Um unser Training möglichst effizient zu halten, wollen wir eine GPU anstelle einer CPU verwenden (letztere kann mehrere Stunden benötigen).

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

# Training loop

Die tqdm Bibliothek bietet uns die Möglichkeit unseren Fortschritt zu veranschaulichen.

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

  0%|          | 0/375 [00:00<?, ?it/s]

# Evaluierung
Wir benötigen nun noch eine Evaluationsfunktion. Anstelle die Metrik am Ende jeder Epoche zu berechnen und zu melden (siehe oben), akkumulieren wir hier alle Batches und berechnen die Metrik am Ende.


In [ ]:
import evaluate

metric = evaluate.load("accuracy")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.582}